In [1]:
import napari
import scipy.ndimage as ndi
import skimage as ski
import glob
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np


In [2]:
viewer = napari.Viewer()

In [3]:
import dask
import cv2
import numpy as np

def backsub(inp, radius=60):
    filterSize =(radius, radius)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                    filterSize)
    blurred = cv2.GaussianBlur(inp, (5, 5), 0)
    tophat_img = cv2.morphologyEx(blurred,
                                cv2.MORPH_TOPHAT,
                                kernel)
    rtn = inp.astype(np.single) - (blurred-tophat_img)
    rtn = np.clip(rtn, 0, np.inf)

    return rtn

def dask_sub(inp, radius=60):
    # Build a list of delayed plane tasks and compute them locally
    img = np.reshape(inp, (-1, inp.shape[-2], inp.shape[-1]))
    tasks = [dask.delayed(backsub)(i, radius) for i in img]
    results = dask.compute(*tasks)  # local compute via wrapper's thread pool
    # results = np.array([backsub(i, radius) for i in img])
    rtn = np.reshape(np.array(results), inp.shape)
    return rtn

# Prepare Mapper

In [4]:
mdf = pd.read_csv('Map.csv')

In [5]:
mdf['MergeWell'] = mdf['Row'] + mdf['Column'].astype('str').str.zfill(2)

In [6]:
good_columns = ['Infection', 'Panel', 'Compound', 'Dose']

In [7]:
for col in good_columns:
    mdf[col] = mdf[col].fillna('')

In [8]:
mdf['Infection'].unique()

<StringArray>
['Non-infected', '', 'DENV-2']
Length: 3, dtype: str

In [9]:
mdf['Panel'].unique()

<StringArray>
['Control', 'Antiviral', 'Immune modulator', 'Toxicity', 'LO (no Mac)',
 'Spare']
Length: 6, dtype: str

In [10]:
mdf['Compound'].unique()

<StringArray>
[    'Untreated',          'DMSO',       'NITD008',    'Sofosbuvir',
   'Mosnodenvir', 'Dexamethasone',   'Baricitinib',        'MCC950',
    'Vitamin D3',      'SB203580', 'Acetaminophen',  'Troglitazone',
    'Amiodarone',            'LO',         'Empty',     'Reference']
Length: 16, dtype: str

In [11]:
mdf['Dose'].unique()

<StringArray>
[     '',   '1uM',  '10uM',  '20uM',   '5uM',  '50uM', '100uM', '0.1nM',
  '25nM', '100nM',   '1nM',  '10nM',   '1mM',   '5mM',  '10mM',  '25uM']
Length: 16, dtype: str

In [12]:
category_orders = {}

category_orders['Panel'] = ['Control', 'Antiviral', 'Immune modulator', 'Toxicity', 'LO (no Mac)','Spare']

category_orders['Infection'] = ['', 'Non-infected', 'DENV-2']

category_orders['Compound'] = [    'Untreated',          'DMSO',       'NITD008',    'Sofosbuvir',
   'Mosnodenvir', 'Dexamethasone',   'Baricitinib',        'MCC950',
    'Vitamin D3',      'SB203580', 'Acetaminophen',  'Troglitazone',
    'Amiodarone',            'LO',         'Empty',     'Reference']

category_orders['Dose'] = ['0.1nM', '1nM', '10nM', '25nM', '100nM', '1uM', '5uM', '10uM', '20uM', '25uM', '50uM', '100uM', '1mM', '5mM', '10mM']

# Merge Data and Mapper

In [13]:
df = pd.read_csv('Quantification.csv')

In [14]:
df['MergeWell'] = df['fname'].str.split('/').str[1].str.split('_').str[0]

In [15]:
df = df.merge(mdf, on='MergeWell')

In [16]:
df.loc[df['MergeWell']=='P24', 'Infection'] = 'DENV-2'

In [17]:
null_count = 1_150_000 / 810 / 810 + 2.0

In [18]:
df['InfectionCompoundDose'] = df['Infection'] + ' | ' + df['Compound'] + ' | ' + df['Dose']


c0_correction = 500_000
c1_correction = 1_150_000

# c0_correction = 0
# c1_correction = 0

df['c0_corrected'] = df['c0'] - c0_correction
df['c1_corrected'] = df['c1'] - c1_correction
df['Ratio'] = df['c1_corrected'] / df['c0_corrected']

df['c1_masked_corrected'] = df['masked_c1']/df['mask'] - null_count



In [19]:
df.to_csv('PerWell.csv')

In [20]:
df.columns

Index(['Unnamed: 0', 'fname', 'time', 'c0', 'c1', 'mask', 'masked_c0',
       'masked_c1', 'MergeWell', 'Well', 'Row', 'Column', 'Infection', 'Panel',
       'Compound', 'Dose', 'Replicate', 'Condition ID',
       'InfectionCompoundDose', 'c0_corrected', 'c1_corrected', 'Ratio',
       'c1_masked_corrected'],
      dtype='str')

# Aggregate by Well

In [21]:
mean_df = df.groupby(['Row', 'Column', 'time']).agg({'c0_corrected':'mean', 'c1_corrected':'mean', 'mask':'mean', 'Ratio':'mean', 'c1_masked_corrected':'mean'}).reset_index()
mean_df.columns = ['Row', 'Column', 'time', 'c0_mean', 'c1_mean', 'mask_mean', 'Ratio_mean', 'c1_masked_corrected_mean',]
mean_df

,Row,Column,time,c0_mean,c1_mean,mask_mean,Ratio_mean,c1_masked_corrected_mean
0,A,1,0,3.210753e+07,65066.129639,52301.0,0.002027,-0.367530
1,A,1,1,2.813837e+07,59056.145264,48005.0,0.002099,-0.350668
2,A,1,2,2.830305e+07,63292.148682,49302.0,0.002236,-0.166669
3,A,1,3,2.687647e+07,55858.983643,46950.0,0.002078,-0.193684
4,A,1,4,2.637325e+07,49951.760010,44592.0,0.001894,-0.311035
...,...,...,...,...,...,...,...,...
5771,P,24,11,1.518015e+07,682910.296631,5089.0,0.044987,29.153542
5772,P,24,12,1.502445e+07,691467.750488,4961.0,0.046023,29.547037
5773,P,24,13,1.491357e+07,700105.666992,4921.0,0.046944,28.912401
5774,P,24,14,1.483843e+07,724771.881104,4615.0,0.048844,30.443723


In [22]:
f = px.line(mean_df, x='time', y='c0_mean', color='Column', facet_col='Row', facet_col_wrap=4, height=800, title='nucSpot Intensity')
f.write_html('nucSpot_Intensity.html')

In [23]:
f = px.line(mean_df, x='time', y='mask_mean', color='Column', facet_col='Row', facet_col_wrap=4, height=800, title='Organoid Quantity')
f.write_html('MaskedOrganoid_Quantity.html')

In [24]:
f = px.line(mean_df, x='time', y='c1_masked_corrected_mean', color='Column', facet_col='Row', facet_col_wrap=4, height=800, title='mScarlet Intensity')
f.write_html('mScarlet_Intensity.html')

# Aggregate and Plot

In [25]:
import scipy.stats as stats

mean_df = df.groupby(['InfectionCompoundDose', 'Infection', 'Panel', 'Compound', 'Dose', 'time']).agg({'c0_corrected':['mean', 'std'], 'c1_corrected':['mean', 'std'], 'Ratio':['mean', 'std'], 'c1_masked_corrected':['mean', 'std'], 'mask':['mean', 'std']}).reset_index()
mean_df.columns = ['InfectionCompoundDose', 'Infection', 'Panel', 'Compound', 'Dose', 'time', 'c0_corrected', 'c0_std', 'c1_corrected', 'c1_std', 'Ratio', 'Ratio_std', 'c1_masked_corrected', 'c1_masked_std', 'mask', 'mask_std']
mean_df

,InfectionCompoundDose,Infection,Panel,Compound,Dose,time,c0_corrected,c0_std,c1_corrected,c1_std,Ratio,Ratio_std,c1_masked_corrected,c1_masked_std,mask,mask_std
0,DENV-2 | Acetaminophen | 10mM,DENV-2,Toxicity,Acetaminophen,10mM,0,4.245112e+06,1.284366e+06,-1.927227e+05,1.118967e+05,-0.055786,0.044150,0.377482,0.583574,29936.2,10676.177345
1,DENV-2 | Acetaminophen | 10mM,DENV-2,Toxicity,Acetaminophen,10mM,1,4.019771e+06,7.532688e+05,-1.515004e+05,1.146273e+05,-0.039865,0.029136,2.269330,1.084401,31065.2,9852.401799
2,DENV-2 | Acetaminophen | 10mM,DENV-2,Toxicity,Acetaminophen,10mM,2,6.473048e+06,1.551303e+06,8.441749e+04,1.336555e+05,0.014871,0.023275,8.478204,3.792475,32005.8,11360.756168
3,DENV-2 | Acetaminophen | 10mM,DENV-2,Toxicity,Acetaminophen,10mM,3,1.044164e+07,2.872234e+06,8.372243e+05,6.512656e+05,0.100135,0.096217,30.773820,14.864024,31583.6,10484.315943
4,DENV-2 | Acetaminophen | 10mM,DENV-2,Toxicity,Acetaminophen,10mM,4,1.296532e+07,4.148605e+06,3.264077e+06,1.934629e+06,0.296663,0.233022,87.760368,45.304958,35362.6,7651.668432
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1163,Non-infected | Vitamin D3 | 1nM,Non-infected,Immune modulator,Vitamin D3,1nM,11,5.652091e+05,1.277103e+04,9.503098e+03,2.720467e+04,0.017386,0.048980,-0.937414,0.212477,18934.0,6584.975740
1164,Non-infected | Vitamin D3 | 1nM,Non-infected,Immune modulator,Vitamin D3,1nM,12,5.593002e+05,1.072032e+04,-2.468114e+03,9.163374e+03,-0.004583,0.016288,-0.898952,0.199110,18363.6,6724.677561
1165,Non-infected | Vitamin D3 | 1nM,Non-infected,Immune modulator,Vitamin D3,1nM,13,5.655577e+05,1.738889e+04,-4.461727e+03,1.654035e+04,-0.008467,0.028432,-0.946867,0.172303,19004.8,7300.908930
1166,Non-infected | Vitamin D3 | 1nM,Non-infected,Immune modulator,Vitamin D3,1nM,14,5.522459e+05,1.310679e+04,-2.157310e+03,1.453782e+04,-0.004160,0.026029,-0.953947,0.172080,18908.8,6769.387099


In [26]:
mean_df.to_csv('Aggregated.csv')

In [57]:
viewer = napari.Viewer()

In [67]:
import plotly.graph_objs as go
import plotly.express as px

f=go.FigureWidget(
    px.line(mean_df[mean_df['Infection']!=''], x='time', y='c1_masked_corrected', facet_col='Compound', facet_row='Infection', color='Dose', height=800, #error_y='c1_masked_std', 
            hover_data=['InfectionCompoundDose', 'Panel'], width=1800, )
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))


for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(4) # This is now necessary as a result of some weird rendering race condition
#f.write_html('FullNuSpot.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Acetaminophen | 10mM', 'Toxicity'],
                                   ['DENV-2 | Acetaminophen | 10mM', 'Toxicity'],
                                   ['DENV-2 | Acetaminophen | 10mM', 'Toxicity'],
                                   ['DENV-2 | Acetaminophen | 10mM', 'Toxicity'],
                                   ['DENV-2 | Acetaminophen | 10mM', 'Toxicity'],
                                   ['DENV-2 | Acetaminophen | 10mM', 'Toxicity'],
                                   ['DENV-2 | Acetaminophen | 10mM', 'Toxicity'],
                                   ['DENV-2 | Acetaminophen | 10mM', 'Toxicity'],
                                   ['DENV-2 | Acetaminophen | 10mM', 'Toxicity'],
                                   ['DENV-2 | Acetaminophen | 10mM', 'Toxicity'],
                                   ['DENV-2 | Acetaminophen | 10mM', 'Toxicity'],
                                   ['DENV-2 | Acetaminophen | 10mM', 'Toxicity'],
 

# Prepping Figures

In [61]:
compound_orders = {'Compound':['Reference', 'Untreated', 'DMSO', 'LO']}

## mScarlet Density

In [62]:
val_to_plot = 'c1_masked_corrected'
std_to_plot = 'c1_masked_std'
range_y = [-10,380]

In [68]:
current_panel = 'Toxicity'

panels = [current_panel, 'Control', 'LO (no Mac)']
cur_df = mean_df[mean_df['Panel'].isin(panels)]

f=go.FigureWidget(
    px.line(cur_df, x='time', y=val_to_plot, facet_col='Compound', facet_row='Infection', color='Dose', height=800, hover_data=['InfectionCompoundDose', 'Panel'], 
            width=1800, range_y=range_y, category_orders=compound_orders, 
            title=f'Panel: {current_panel}')
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))



for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(4) # This is now necessary as a result of some weird rendering race condition
f.write_html(current_panel + '.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   [

In [31]:
current_panel = 'Toxicity'

panels = [current_panel, 'Control', 'LO (no Mac)']
cur_df = mean_df[mean_df['Panel'].isin(panels)]

f=go.FigureWidget(
    px.line(cur_df, x='time', y=val_to_plot, facet_col='Compound', facet_row='Infection', color='Dose', height=800, hover_data=['InfectionCompoundDose', 'Panel'], error_y=std_to_plot,
            width=1800, range_y=range_y, category_orders=compound_orders, 
            title=f'Panel: {current_panel}')
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))

for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(2) # This is now necessary as a result of some weird rendering race condition
f.write_html(current_panel + '_error.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   [

In [32]:
current_panel = 'Antiviral'

panels = [current_panel, 'Control', 'LO (no Mac)']
cur_df = mean_df[mean_df['Panel'].isin(panels)]

f=go.FigureWidget(
    px.line(cur_df, x='time', y=val_to_plot, facet_col='Compound', facet_row='Infection', color='Dose', height=800, hover_data=['InfectionCompoundDose', 'Panel'], 
            width=1800, range_y=range_y, category_orders=compound_orders, 
            title=f'Panel: {current_panel}')
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))

for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(2) # This is now necessary as a result of some weird rendering race condition
f.write_html(current_panel + '.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   [

In [33]:
current_panel = 'Antiviral'

panels = [current_panel, 'Control', 'LO (no Mac)']
cur_df = mean_df[mean_df['Panel'].isin(panels)]

f=go.FigureWidget(
    px.line(cur_df, x='time', y=val_to_plot, facet_col='Compound', facet_row='Infection', color='Dose', height=800, hover_data=['InfectionCompoundDose', 'Panel'], error_y=std_to_plot,
            width=1800, range_y=range_y, category_orders=compound_orders, 
            title=f'Panel: {current_panel}')
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))

for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(2) # This is now necessary as a result of some weird rendering race condition
f.write_html(current_panel + '_error.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   [

In [34]:
current_panel = 'Immune modulator'

panels = [current_panel, 'Control', 'LO (no Mac)']
cur_df = mean_df[mean_df['Panel'].isin(panels)]

f=go.FigureWidget(
    px.line(cur_df, x='time', y=val_to_plot, facet_col='Compound', facet_row='Infection', color='Dose', height=800, hover_data=['InfectionCompoundDose', 'Panel'], 
            width=1800, range_y=range_y, category_orders=compound_orders, 
            title=f'Panel: {current_panel}')
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))

for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(2) # This is now necessary as a result of some weird rendering race condition
f.write_html(current_panel + '.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   [

In [35]:
current_panel = 'Immune modulator'

panels = [current_panel, 'Control', 'LO (no Mac)']
cur_df = mean_df[mean_df['Panel'].isin(panels)]

f=go.FigureWidget(
    px.line(cur_df, x='time', y=val_to_plot, facet_col='Compound', facet_row='Infection', color='Dose', height=800, hover_data=['InfectionCompoundDose', 'Panel'], error_y=std_to_plot,
            width=1800, range_y=range_y, category_orders=compound_orders, 
            title=f'Panel: {current_panel}')
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))

for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(2) # This is now necessary as a result of some weird rendering race condition
f.write_html(current_panel + '_error.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   [

## Organoid Quantity

In [43]:
val_to_plot = 'mask'
std_to_plot = 'mask_std'
range_y = [-10,80_000]

In [44]:
current_panel = 'Toxicity'

panels = [current_panel, 'Control', 'LO (no Mac)']
cur_df = mean_df[mean_df['Panel'].isin(panels)]

f=go.FigureWidget(
    px.line(cur_df, x='time', y=val_to_plot, facet_col='Compound', facet_row='Infection', color='Dose', height=800, hover_data=['InfectionCompoundDose', 'Panel'], 
            width=1800, category_orders=compound_orders, 
            title=f'Panel: {current_panel}')
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))



for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(2) # This is now necessary as a result of some weird rendering race condition
f.write_html(current_panel + '_organoid.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   [

In [45]:
current_panel = 'Toxicity'

panels = [current_panel, 'Control', 'LO (no Mac)']
cur_df = mean_df[mean_df['Panel'].isin(panels)]

f=go.FigureWidget(
    px.line(cur_df, x='time', y=val_to_plot, facet_col='Compound', facet_row='Infection', color='Dose', height=800, hover_data=['InfectionCompoundDose', 'Panel'], error_y=std_to_plot,
            width=1800, range_y=range_y, category_orders=compound_orders, 
            title=f'Panel: {current_panel}')
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))

for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(2) # This is now necessary as a result of some weird rendering race condition
f.write_html(current_panel + '_organoid_error.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   [

In [46]:
current_panel = 'Antiviral'

panels = [current_panel, 'Control', 'LO (no Mac)']
cur_df = mean_df[mean_df['Panel'].isin(panels)]

f=go.FigureWidget(
    px.line(cur_df, x='time', y=val_to_plot, facet_col='Compound', facet_row='Infection', color='Dose', height=800, hover_data=['InfectionCompoundDose', 'Panel'], 
            width=1800, range_y=range_y, category_orders=compound_orders, 
            title=f'Panel: {current_panel}')
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))

for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(2) # This is now necessary as a result of some weird rendering race condition
f.write_html(current_panel + '_organoid.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   [

In [47]:
current_panel = 'Antiviral'

panels = [current_panel, 'Control', 'LO (no Mac)']
cur_df = mean_df[mean_df['Panel'].isin(panels)]

f=go.FigureWidget(
    px.line(cur_df, x='time', y=val_to_plot, facet_col='Compound', facet_row='Infection', color='Dose', height=800, hover_data=['InfectionCompoundDose', 'Panel'], error_y=std_to_plot,
            width=1800, range_y=range_y, category_orders=compound_orders, 
            title=f'Panel: {current_panel}')
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))

for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(2) # This is now necessary as a result of some weird rendering race condition
f.write_html(current_panel + '_organoid_error.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   [

In [48]:
current_panel = 'Immune modulator'

panels = [current_panel, 'Control', 'LO (no Mac)']
cur_df = mean_df[mean_df['Panel'].isin(panels)]

f=go.FigureWidget(
    px.line(cur_df, x='time', y=val_to_plot, facet_col='Compound', facet_row='Infection', color='Dose', height=800, hover_data=['InfectionCompoundDose', 'Panel'], 
            width=1800, range_y=range_y, category_orders=compound_orders, 
            title=f'Panel: {current_panel}')
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))

for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(2) # This is now necessary as a result of some weird rendering race condition
f.write_html(current_panel + '_organoid.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   [

In [49]:
current_panel = 'Immune modulator'

panels = [current_panel, 'Control', 'LO (no Mac)']
cur_df = mean_df[mean_df['Panel'].isin(panels)]

f=go.FigureWidget(
    px.line(cur_df, x='time', y=val_to_plot, facet_col='Compound', facet_row='Infection', color='Dose', height=800, hover_data=['InfectionCompoundDose', 'Panel'], error_y=std_to_plot,
            width=1800, range_y=range_y, category_orders=compound_orders, 
            title=f'Panel: {current_panel}')
    )

def click_fn(trace, points, state):

    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        all_imgs = []
        all_labels = []
        for fname in df[df['InfectionCompoundDose']==idx]['fname'].unique():
            img = ski.io.imread(fname.replace('_montage.tif', '_montage_ds.tiff'))
            img = ski.transform.downscale_local_mean(img, (1, 4, 4, 1))
            img[:,:,:,0] = dask_sub(img[:,:,:,0], radius=60)
            img[:,:,:,1] = dask_sub(img[:,:,:,1], radius=60)
            all_imgs.append(img)

            labels = ski.io.imread(fname.replace('_montage.tif', '_organoid_labels_ds.tiff'))
            all_labels.append(labels)
        viewer.layers.clear()
        viewer.add_image(np.concatenate(all_imgs, axis=2), channel_axis=-1)
        viewer.add_labels(np.concatenate(all_labels, axis=2), scale=(1,.25,.25))

for a in f.data:
    a.on_click(click_fn)
import time
time.sleep(2) # This is now necessary as a result of some weird rendering race condition
f.write_html(current_panel + '_organoid_error.html')
f

FigureWidget({
    'data': [{'customdata': array([['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   ['DENV-2 | Reference | ', 'Control'],
                                   [